# ex01 · 激活函数（对应教材 4.2 激活函数）

> **做题流程**：先预测（曲线长什么样、饱和区在哪），再运行验证。
> **做完再看** `solutions/ex01-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）
>
> 本节从零实现三种激活函数并观察曲线与梯度，理解「为什么需要非线性」和「为什么 ReLU 主流」（面试高频）。

In [2]:
%matplotlib inline
import torch
import matplotlib.pyplot as plt

## 题 1 🌱 从零实现三个激活函数（TODO 1.1 ~ 1.3）

先补全三个函数，再在【你的预测】里写下：relu / sigmoid / tanh 的输出范围分别是什么？

In [4]:
help(torch.zeros_like)

Help on built-in function zeros_like in module torch:

zeros_like(...)
    zeros_like(input, *, dtype=None, layout=None, device=None, requires_grad=False, memory_format=torch.preserve_format) -> Tensor
    
    Returns a tensor filled with the scalar value `0`, with the same size as
    :attr:`input`. ``torch.zeros_like(input)`` is equivalent to
    ``torch.zeros(input.size(), dtype=input.dtype, layout=input.layout, device=input.device)``.
    
    .. warning::
        As of 0.4, this function does not support an :attr:`out` keyword. As an alternative,
        the old ``torch.zeros_like(input, out=output)`` is equivalent to
        ``torch.zeros(input.size(), out=output)``.
    
    Args:
        input (Tensor): the size of :attr:`input` will determine size of the output tensor.
    
    Keyword args:
        dtype (:class:`torch.dtype`, optional): the desired data type of returned Tensor.
            Default: if ``None``, defaults to the dtype of :attr:`input`.
        layout (:class:

In [5]:
help(torch.max)

Help on built-in function max in module torch:

max(...)
    max(input, *, out=None) -> Tensor
    
    Returns the maximum value of all elements in the ``input`` tensor.
    
    .. note::
        The difference between ``max``/``min`` and ``amax``/``amin`` is:
            - ``amax``/``amin`` supports reducing on multiple dimensions,
            - ``amax``/``amin`` does not return indices.
    
        Both ``amax``/``amin`` evenly distribute gradients between equal values
        when there are multiple input elements with the same minimum or maximum value.
    
        For ``max``/``min``:
            - If reduce over all dimensions(no dim specified), gradients evenly distribute between equally ``max``/``min`` values.
            - If reduce over one specified axis, only propagate to the indexed element.
    
    Args:
        input (Tensor): the input tensor.
    
    Keyword args:
        out (Tensor, optional): the output tensor.
    
    Example::
    
        >>> a = torch.rand

In [14]:
def relu(X):
    # TODO 1.1: max(X, 0)。提示 torch.max(X, torch.zeros_like(X))
    return torch.max(X, torch.zeros_like(X))


def sigmoid(X):
    # TODO 1.2: 1 / (1 + exp(-X))
    return 1 / (1 + torch.exp(-X))


def tanh(X):
    # TODO 1.3: (exp(X) - exp(-X)) / (exp(X) + exp(-X))
    return (torch.exp(X) - torch.exp(-X)) / (torch.exp(X) + torch.exp(-X))

**【你的预测】**

- relu 输出范围：(0, X)　（负数输入输出什么？）
- sigmoid 输出范围：(0, 1)
- tanh 输出范围：(-1, 1)　（和 sigmoid 差在哪？）

tanh 和 sigmoid 的导数确实不同（tanh 最大 1.0 vs sigmoid 0.25，梯度消失慢 4 倍），0 附近梯度也不同。但更本质的差别是零中心：sigmoid 输出恒正导致下一层梯度方向单一（zigzag、收敛慢），tanh 输出有正有负、梯度平衡、收敛快。所以隐藏层优先用 tanh，sigmoid 更多用在二分类输出层（因为它输出 (0,1) 天然是概率）。

In [15]:
try:
    assert relu(torch.tensor([-2.0, 0.0, 2.0])).tolist() == [0.0, 0.0, 2.0]
    assert abs(sigmoid(torch.tensor([0.0])).item() - 0.5) < 1e-6
    assert abs(tanh(torch.tensor([0.0])).item()) < 1e-6
    print('✓ 三个激活函数数值正确')
except NotImplementedError as e:
    print(f'⚠ {e}')
except AssertionError as e:
    print(f'✗ {e}')

✓ 三个激活函数数值正确


## 题 2 🔧 画曲线，观察形状

先手画三条约略曲线（哪些过原点？哪些两端变平？），再运行对照。

**【你的预测】**

In [ ]:
x = torch.arange(-8.0, 8.0, 0.1)
plt.figure(figsize=(8, 3.5))
plt.plot(x.numpy(), relu(x).numpy(), label='relu')
plt.plot(x.numpy(), sigmoid(x).numpy(), label='sigmoid')
plt.plot(x.numpy(), tanh(x).numpy(), label='tanh')
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.legend()
plt.title('三种激活函数')
plt.show()

## 题 3 🔧 梯度饱和（面试高频）

先预测：sigmoid 在 x=5 和 x=10 处的导数（斜率）大约是多少？为什么曲线两端「变平」会导致问题？

x 越大（或越小），sigmoid 越接近 0 或 1，导数越趋近 0 → 反向传播时梯度逐层连乘，多个饱和 sigmoid 叠加 → **梯度消失**。

In [16]:
x = torch.tensor([5.0, 10.0], requires_grad=True)
y = sigmoid(x).sum()
y.backward()
print('sigmoid 在 x=5  的梯度:', round(x.grad[0].item(), 6))
print('sigmoid 在 x=10 的梯度:', round(x.grad[1].item(), 6))
# 提示: sigmoid'(x) = sigmoid(x) * (1 - sigmoid(x))，x 越大越接近 0

sigmoid 在 x=5  的梯度: 0.006648
sigmoid 在 x=10 的梯度: 4.5e-05


## 题 4 🔧 问答：为什么 ReLU 是主流？（面试高频）

先写你的理解，再对照答案文件：

1. 为什么需要非线性激活函数？如果网络里没有激活函数会怎样？

没有非线性的激活函数，矩阵相乘仍然是矩阵，导致多层网络实则是线性函数，多少层都实则为一层，整个网络发生塌陷。且线性函数难以拟合复杂函数

2. 相比 sigmoid/tanh，ReLU 有什么优势？

方便求导，开销少
（核心）ReLU 正半轴梯度恒为 1，不饱和，反向传播时梯度不衰减 → 训练快。sigmoid/tanh 两端饱和 → 梯度消失。所以 ReLU 成为隐藏层的主流选择。

3. 什么是 Dying ReLU？怎么缓解？

当x < 0 时，函数值为0，导数值为0，该神经元在之后就不会发生任何更新，形象的称之为dying relu。 一般的做法用leaky relu，在小于0的地方给一个极小的梯度，如y = 0.01x
or PReLU（斜率可学习）、用较小学习率。

**【你的预测】**

## 小结与面试衔接

- 没有激活函数：多层线性 = 一层线性，深度失去意义
- sigmoid/tanh 两端饱和 → 梯度消失；ReLU 正半轴梯度恒 1 → 训练快，成为主流
- Dying ReLU：负半轴梯度为 0，神经元可能「死掉」；缓解：Leaky ReLU / 较小学习率
- 一轮「骨架组件」考点：激活函数的种类、特点、意义、为什么需要非线性、Dying ReLU